### 1. Replace all instances of an incorrectly typed location. The mistakes come from (a) lack of capitalization and (b) a missing dash "-".

In [ ]:
# (a) capitalization → handled by (?i)
# (b) missing dash → handled by [- ] (space or dash both normalized to -).

# pattern: "new york" or "new-york", any capitalization
pattern = r"(?i)new[- ]york"
df["reviews"] = df["reviews"].str.replace(pattern, "New-York", regex=True)

#What this does:

# (?i) → ignore case (so new, New, NEW all match)
# [- ] → either a dash or a space
# york → the second word
# str.replace(..., regex=True) → replaces all matches of that regex with "New-York".

pattern = r"(?i)los[- ]angeles"
df["reviews"] = df["reviews"].str.replace(pattern, "Los-Angeles", regex=True)


### 2. Create a new DataFrame with only the reviews containing a given string pattern. The string pattern will involve one or more special characters from regular expressions.

In [ ]:
pattern = r"\b\d{5}\b"   # example pattern

# You just swap pattern for whatever they give you, e.g.

# words starting with “un”: r"\bun\w*"
# email-like strings: r"\S+@\S+\.\S+"#

df_match = df[df["reviews"].str.contains(pattern, regex=True, na=False)].copy()


### 3. Find all occurrences in the reviews column of two words with similar spellings. You will also need to include both the plural and singular versions of the word. The final result should only include cases where a match was found.

### Goal: in "reviews", find all appearances of two similar words (e.g. color/colour, cat/kat, etc.), including plural versions, and drop rows with no matches.

In [ ]:
# Example: "color" vs "colour", singular + plural
pattern = r"\b(?:color|colors|colour|colours)\b"

word_matches = df["reviews"].str.findall(pattern)

# keep only rows with at least one match
word_matches = word_matches[word_matches.str.len() > 0]

# More compact pattern using s? for optional plural:
pattern = r"\b(?:color|colour)s?\b"
word_matches = df["reviews"].str.findall(pattern)
word_matches = word_matches[word_matches.str.len() > 0]

# str.findall → list of all matches per review.
 #.str.len() > 0 → filters to rows where there is at least one match.

### 4. Find all occurrences in the reviews column of words ending with a given letter of the alphabet. The final result should only include cases where a match was found.

In [ ]:
# Suppose the letter is e. General template:
letter = "e"  # change this on the quiz

pattern = rf"\b\w*{letter}\b"    # word boundary, any word ending in that letter
pattern = rf"\b\w*e\b"  

end_matches = df["reviews"].str.findall(pattern)
end_matches = end_matches[end_matches.str.len() > 0]

words ending in y: r"\b\w*y\b"
words ending in s: r"\b\w*s\b"

### 5. Create a new column of the DataFrame called "tokens" where each string in the reviews column is converted to a list of individual lower-case words.

In [ ]:
# Simplest pattern: lowercase first, then find words

word_lists = df["reviews"].str.findall(r"[A-Za-z']+")

def to_lower_list(words):
    return [w.lower() for w in words]

df["tokens"] = word_lists.apply(to_lower_list)

# the code below does the same thing:

df["tokens"] = df["reviews"].str.lower().str.findall(r"[a-z']+")


### 6. Unravel the tokens column into a single Pandas Series containing all the words appearing in the all the reviews. Create a new Pandas series with the probability distribution of the words appearing the in the DataSet. Then create a bar plot of the probabilities for a subset of the most common words.

In [ ]:
# a) Unravel / flatten the tokens
all_words = df["tokens"].explode()
# Now all_words is a Series like:
0   this
0   movie
0   was
1   great
1   movie
...

# b) Probability distribution of words
word_probs = all_words.value_counts(normalize=True)

# c) Bar plot for the most common words
top_k = 20  # or whatever range the questions ask for, if everything, take out .head()

ax = word_probs.head(top_k).plot(kind="bar")
ax.set_title("Top word probabilities")
ax.set_ylabel("Probability")
ax.set_xlabel("Word")
